# Chapter 27: a small, inspectable representation studio

Read STUDY_CONTRACT.md first. These are invented signals, not EEG measurements.
No default neural results, clinical validity, or performance gains are claimed.
The authoring review was static; execute the cells and explain your own results.
Run this notebook from its chapter folder. The PyTorch cell is optional.


In [ ]:
from pathlib import Path
import copy
import foundation_studio as fs

data = fs.fixture(seed=2701)
records, labels = data['records'], data['labels']
counts = fs.split_audit(records)
counts, len(records)


## 1. Follow people before windows

There are 12 development, six validation, and six evaluation people. Each contributes
four records. Label is deliberately counterbalanced within each person. Explain why
96 windows do not mean 96 independent people. Create an overlap fault and inspect it.


In [ ]:
overlap = copy.deepcopy(records[0])
overlap['record_id'] += '-duplicate-person'
overlap['split'] = 'evaluation'
try:
    fs.split_audit(records + [overlap])
except ValueError as exc:
    print('Expected audit failure:', exc)

dev = [r for r in records if r['split'] == 'development']
normalizer = fs.fit_normalizer(dev)
tokens = fs.tokenize(dev[0], normalizer)
len(tokens['tokens']), len(tokens['tokens'][0]), sum(tokens['valid'])


## 2. Trace the schema

128 Hz × 4 s = 512 samples per channel. Each 32-sample patch lasts 0.25 seconds.
Two channels × 16 patches = 32 tokens. Inspect token 17: which channel and time does
it represent? Explain why the development normalizer is not a personal baseline.


In [ ]:
fault_report = {}
for name, record in fs.fault_cases(dev[0]).items():
    try:
        row = fs.tokenize(record, normalizer)
        fault_report[name] = {'state': 'accepted_with_mask', 'valid_tokens': sum(row['valid'])}
    except ValueError as exc:
        fault_report[name] = {'state': 'rejected', 'reason': str(exc)}
fault_report


## 3. Give a transparent baseline a fair chance

Use exactly the first six development people as labeled examples. The two fixed
frequencies are explicitly matched to the toy recipe; this is not feature discovery
on real EEG. The final cohort must remain outside fitting and selection.


In [ ]:
people = sorted({r['person_id'] for r in dev})[:6]
labeled = [r for r in dev if r['person_id'] in people]
weights = fs.logistic_fit([fs.spectral_features(r) for r in labeled],
                          [labels[r['record_id']] for r in labeled])
baseline_report = {}
for split in ['validation', 'evaluation']:
    rows = [r for r in records if r['split'] == split]
    probabilities = fs.logistic_predict(weights, [fs.spectral_features(r) for r in rows])
    baseline_report[split] = {'metrics': fs.metrics([labels[r['record_id']] for r in rows], probabilities),
                              'record_ids': [r['record_id'] for r in rows],
                              'person_ids': [r['person_id'] for r in rows],
                              'probabilities': probabilities}
baseline_report


## 4. Optional: compare tiny neural representations

Install requirements-torch.txt, then change RUN_TORCH to True. Pretraining uses all
development inputs without labels; classifiers use the same six labeled people.
The random and pretrained probes train only their heads; the scratch model trains
its encoder too. The pretrained route spends extra compute, so this is not a
matched-total-compute comparison. Leave this disabled if PyTorch is unavailable.


In [ ]:
RUN_TORCH = False
neural_report, checkpoint_bundle = None, None
if RUN_TORCH:
    from torch_studio import run_experiment
    neural_report, checkpoint_bundle = run_experiment(data, seed=2701, label_people=6)
    display(neural_report['metrics'])
else:
    print('Optional neural experiment not run; do not report a neural performance result.')


## 5. Inspect evidence without asking a model to invent it

These are author-written teaching passages. Filtering occurs before lexical ranking.
The private, retired, and pre-quarantined passages are excluded for the student role.
This does not demonstrate a production access-control service or automatic injection
detection. Exact text matching checks a narrow quotation relation, not scientific truth.


In [ ]:
evidence = fs.retrieve('tokenizer sample rate patch')
claim_checks = []
if evidence:
    p = evidence[0]
    claim_checks.append(fs.inspect_claim(p['text'], p['id'], evidence))
    claim_checks.append(fs.inspect_claim('This tokenizer diagnoses neurological disease.', p['id'], evidence))
claim_checks.append(fs.inspect_claim('Unsupported statement', 'absent:p1', evidence))
evidence, claim_checks


## 6. Export what another reader can inspect

The export creates a new folder. Keep model states when the optional experiment runs.
The hash receipt covers all other exported files and supports byte comparison, not
authentication. Do not load untrusted checkpoints or commit generated artifacts.


In [ ]:
report = {'baseline': baseline_report, 'baseline_weights': weights,
          'labeled_people': people, 'faults': fault_report,
          'claim_checks': claim_checks, 'neural': neural_report,
          'scope': 'invented teaching task; descriptive metrics; no clinical validity'}
folder = fs.export_bundle(data, normalizer, report, evidence)
if checkpoint_bundle is not None:
    from torch_studio import save_checkpoint
    print(save_checkpoint(checkpoint_bundle, folder))
print(folder)


## Reflection

Explain what the normalizer learned, which data each model fitted, and which claims
the evidence table supports. Relate a pretraining loss change to downstream results
without assuming improvement. State the uncertainty due to six evaluation people,
repeated windows, one seed, and one invented recipe. Propose a different task and an
independent recording source before making a transferable-representation claim.
